# CogniCore-JAX: Entrenamiento en GPU con Google Colab

Este notebook entrena CogniCore usando **JAX** para aprovechar la **GPU** de Google Colab.

## Aceleración esperada
| Dispositivo | Velocidad relativa |
|-------------|-------------------|
| CPU (NumPy original) | 1x |
| GPU T4 (Colab gratuito) | ~10-20x |
| GPU A100 (Colab Pro) | ~30-50x |

## Instrucciones
1. **Runtime > Change runtime type > GPU** (T4 o A100)
2. Ejecuta las celdas en orden
3. Los checkpoints se guardan en Google Drive

## 1. Clonar repositorio y configurar entorno

In [ ]:
# Clonar el repositorio de GitHub automáticamente
!git clone https://github.com/CerberusLD914/Cognicore.git /content/Cognicore

# Verificar que los archivos se clonaron correctamente
!ls -la /content/Cognicore/cognicore_jax/
print("\n✅ Repositorio clonado exitosamente")

In [ ]:
# Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("⚠️  No se detectó GPU. Ve a Runtime > Change runtime type > GPU")

In [ ]:
# Instalar JAX con soporte CUDA
!pip install -U "jax[cuda12]" flax optax -q

import jax
import jax.numpy as jnp
print(f"JAX version: {jax.__version__}")
print(f"Dispositivos: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

In [ ]:
# Montar Google Drive para guardar checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Crear directorio de trabajo
import os
WORK_DIR = '/content/drive/MyDrive/CogniCore_GPU'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
print(f"Directorio de trabajo: {WORK_DIR}")

In [ ]:
# Configurar el path para importar los módulos
import sys
sys.path.insert(0, '/content/Cognicore')

# Verificar que los módulos JAX existen
from cognicore_jax.model import build
from cognicore_jax.train import train
print("✅ Módulos importados correctamente")

## 2. Entrenamiento en GPU

In [ ]:
# OBLIGATORIO antes de entrenar: comprobar que JAX y NumPy dan la misma
# pérdida. Si divergen, entrenar 10000 pasos es tiempo desperdiciado.
!cd /content/Cognicore && python verify_jax.py

# Configuración del entrenamiento
import os
os.chdir(WORK_DIR)

# Parámetros
TARGET_PARAMS = 10_000_000  # 10M parámetros
CORPUS_MB = 8              # 8 MB de corpus
STEPS = 2000               # Pasos de entrenamiento
BATCH = 8                  # Tamaño de batch
SEQ_LEN = 256              # Longitud de secuencia
LR = 3e-3                  # Learning rate
EVAL_EVERY = 200           # Evaluar cada N pasos
SEED = 0

print(f"Configuración:")
print(f"  Parámetros objetivo: {TARGET_PARAMS:,}")
print(f"  Corpus: {CORPUS_MB} MB")
print(f"  Pasos: {STEPS}")
print(f"  Batch: {BATCH}")
print(f"  Secuencia: {SEQ_LEN}")
print(f"  Learning rate: {LR}")

In [ ]:
# Entrenamiento
from cognicore_jax.train import train

state, model, cfg, hist, best = train(
    target_params=TARGET_PARAMS,
    corpus_bytes=CORPUS_MB * 1_000_000,
    steps=STEPS,
    batch=BATCH,
    seq_len=SEQ_LEN,
    lr=LR,
    out="cognicore-10M",
    seed=SEED,
    allow_network=True,
    eval_every=EVAL_EVERY,
    checkpoint_dir=f"{WORK_DIR}/checkpoints"
)

## 3. Evaluación y generación

In [ ]:
# Evaluación final
import jax.numpy as jnp
import numpy as np
from cognicore_jax.model import cross_entropy
from cognicore_jax.data import load_corpus, to_byte_tokens, causal_lm_batch

# Cargar corpus de validación
text = load_corpus(2_000_000, log=lambda *a: None)
data = to_byte_tokens(text, SEQ_LEN)
val = data[:256]
val_ids = jnp.array(val[:, :-1])
val_tgt = jnp.array(val[:, 1:])

from cognicore_jax.train import eval_step
final_loss = float(eval_step(state, val_ids, val_tgt))
print(f"Loss final: {final_loss:.4f}")
print(f"Bits/byte: {final_loss / np.log(2):.4f}")
print(f"Compresión: {np.log2(256) / (final_loss / np.log(2)):.2f}x mejor que uniforme")

In [ ]:
# Generación de texto
from cognicore_jax.model import CogniCore, local_context, l2_normalize
import jax.numpy as jnp

def generate(model, state, prompt="", n=200, temperature=0.85, top_k=40, seed=0):
    """Genera texto byte a byte."""
    rng = np.random.default_rng(seed)
    chunk = model.cfg.seq_len
    
    pre = np.frombuffer(prompt.encode('utf-8'), dtype=np.uint8).astype(np.int64)
    if len(pre) > chunk - 2:
        pre = pre[-(chunk - 2):]
    
    win = np.tile(pre, (1, 1)).astype(np.int64) if len(pre) else np.full((1, 1), 10, dtype=np.int64)
    out = []
    
    for _ in range(n):
        ids = jnp.array(win)
        logits = state.apply_fn({"params": state.params}, ids, refine=True)
        lg = np.array(logits[0, -1, :])
        
        # Temperature
        lg = lg / max(temperature, 1e-3)
        
        # Top-k
        if top_k and top_k < len(lg):
            kth = np.partition(lg, -top_k)[-top_k]
            lg = np.where(lg < kth, -1e9, lg)
        
        # Softmax
        p = np.exp(lg - lg.max())
        p /= p.sum()
        
        nxt = rng.choice(len(p), p=p)
        out.append(nxt)
        
        win = np.concatenate([win, np.array([[nxt]], dtype=np.int64)], axis=1)
        if win.shape[1] > chunk:
            win = win[:, -chunk:]
    
    return bytes(x if x < 256 else 10 for x in out).decode('utf-8', 'replace')

# Generar con diferentes prompts
print("Generación greedy (argmax):")
print("-" * 60)
for prompt in ["def add(a, b):\n", "for i in range(10):\n", "import "]:
    result = generate(model, state, prompt=prompt, n=100, temperature=0.7, top_k=20, seed=0)
    print(f"Prompt: {repr(prompt)}")
    print(f"Resultado: {repr(result)}")
    print()

## 4. Guardar y exportar

In [ ]:
# Guardar checkpoint en Google Drive
import shutil
from pathlib import Path

ck_dir = Path(f"{WORK_DIR}/checkpoints")
print("Checkpoints guardados:")
for f in ck_dir.glob("*"):
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")

# Descargar checkpoint (opcional)
from google.colab import files
for f in ck_dir.glob("*.npz"):
    files.download(f)
    print(f"Descargado: {f.name}")

## 5. Entrenamiento continuo (opcional)

Para continuar el entrenamiento desde un checkpoint:

In [ ]:
# Entrenamiento continuo desde checkpoint
# Descomenta y modifica los parámetros según necesites

# state, model, cfg, hist, best = train(
#     target_params=TARGET_PARAMS,
#     corpus_bytes=CORPUS_MB * 1_000_000,
#     steps=STEPS * 2,  # El doble de pasos
#     batch=BATCH,
#     seq_len=SEQ_LEN,
#     lr=LR * 0.1,  # LR más bajo para fine-tuning
#     out="cognicore-10M",
#     seed=SEED,
#     allow_network=True,
#     eval_every=EVAL_EVERY,
#     resume=True,
#     checkpoint_dir=f"{WORK_DIR}/checkpoints"
# )